In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error as sklearn_mae, r2_score
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
target_column = 'Delivery_Time'
plt.hist(df[target_column], edgecolor='black')

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
print(df.isnull().sum())
print(df.isnull().sum().sum())

#There's 311 missing values
null_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']

# All of our features here are important AND if the target is missing we cant predict
# DROP ALL

df_clean = df.dropna(subset=null_cols)
print(df_clean)
print(df_clean.isnull().sum().sum())

#Job complete

In [ ]:
# Task 3: Write your code here:

# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
cat_cols = list(df_clean.select_dtypes(include=["object"]).columns)
print(cat_cols)

label_encoder = LabelEncoder()
df_enc = df_clean

for col in cat_cols:
    df_enc[col] = label_encoder.fit_transform(df_enc[col])

print(df_enc.head())

In [ ]:
# Task 5: Write your code here:
features = df_enc.columns.drop(["Delivery_Time", 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'])  # DON'T SCALE THE TARGET

df_scaled = df_enc
scaler = StandardScaler()
df_scaled[features] = scaler.fit_transform(df_scaled[features])

df_scaled.head()

In [ ]:
# Task 6: Write your code here:
#This was done above, but will be re-done here

plt.hist(df_scaled[target_column], edgecolor='black')

#The target is not balanced, but it has a standard distribution, and its slightly right skewed
#However we are dealing with regression here, so balance is not a major concern

In [ ]:
# Task 1: Write your code here:
X = df_scaled.drop("Delivery_Time", axis=1).astype(float)
y = df_scaled['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}


n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': [], 'mse': [], 'rmse': [], 'r2': []}


for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = sklearn_mae(y_test, y_pred)
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mae"].append(mae)
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)


for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance

# Gather importances from the models (from the last fold)
importances = {}

importances['Random Forest'] = models['Random Forest Regressor'].feature_importances_
importances['CatBoost'] = models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(models['Random Forest Regressor'].predictions, bins=50, edgecolor='black')

#This is something like how it would work but honestly im running out of time to hunt this correct syntax down

In [ ]:
# Task Bonus: Write your code here:
# I did something along the lines of 2/3rds of it above, please check